#### Инициализация

In [1]:
from data.MNIST3d.voxelgrid import VoxelGrid

In [2]:
train_file = "./data/MNIST3d/train_point_clouds.h5"

In [3]:
import utils.subsampling
import importlib

importlib.reload(utils.subsampling)

point_clouds, labels = utils.subsampling.subsample_dataset(train_file, max_samples_per_class=50)

#### Извлечение признаков

In [4]:
import utils.alpha_persistence
importlib.reload(utils.alpha_persistence)

<module 'utils.alpha_persistence' from 'c:\\Users\\USER\\Desktop\\topology-mnist3d\\utils\\alpha_persistence.py'>

In [5]:
from sklearn.pipeline import make_pipeline, make_union
from gtda.diagrams import PersistenceEntropy, Amplitude
from gtda.images import Padder

# Creating the diagram generation pipeline
diagram_steps = [
    [
        utils.alpha_persistence.AlphaComplexTransformer(n_jobs=-1),
        #Padder()
    ]
]

# Listing all metrics we want to use to extract diagram amplitudes
metric_list = [
    {"metric": "bottleneck", "metric_params": {}},
    {"metric": "wasserstein", "metric_params": {"p": 1}},
    {"metric": "wasserstein", "metric_params": {"p": 2}},
    {"metric": "landscape", "metric_params": {"p": 1, "n_layers": 1, "n_bins": 100}},
    {"metric": "landscape", "metric_params": {"p": 1, "n_layers": 2, "n_bins": 100}},
    {"metric": "landscape", "metric_params": {"p": 2, "n_layers": 1, "n_bins": 100}},
    {"metric": "landscape", "metric_params": {"p": 2, "n_layers": 2, "n_bins": 100}},
    {"metric": "betti", "metric_params": {"p": 1, "n_bins": 100}},
    {"metric": "betti", "metric_params": {"p": 2, "n_bins": 100}},
    {"metric": "heat", "metric_params": {"p": 1, "sigma": 1.6, "n_bins": 100}},
    {"metric": "heat", "metric_params": {"p": 1, "sigma": 3.2, "n_bins": 100}},
    {"metric": "heat", "metric_params": {"p": 2, "sigma": 1.6, "n_bins": 100}},
    {"metric": "heat", "metric_params": {"p": 2, "sigma": 3.2, "n_bins": 100}},
]

#
feature_union = make_union(
    *[PersistenceEntropy(nan_fill_value=-1)]
    + [Amplitude(**metric, n_jobs=-1) for metric in metric_list],
    n_jobs=-1
)

tda_union = make_union(
    *[make_pipeline(*diagram_step, feature_union) for diagram_step in diagram_steps],
    n_jobs=-1
)

In [6]:
from sklearn import set_config
set_config(display='diagram')

tda_union

FeatureUnion(n_jobs=-1,
             transformer_list=[('pipeline',
                                Pipeline(steps=[('alphacomplextransformer',
                                                 AlphaComplexTransformer()),
                                                ('featureunion',
                                                 FeatureUnion(n_jobs=-1,
                                                              transformer_list=[('persistenceentropy',
                                                                                 PersistenceEntropy(nan_fill_value=-1)),
                                                                                ('amplitude-1',
                                                                                 Amplitude(metric='bottleneck',
                                                                                           metric_params={},
                                                                                           n_jobs=-1)),
                                                                                ('amplitude-2',
                                                                                 Amplitude(metric='wa...
                                                                                           metric_params={'n_bins': 100,
                                                                                                          'p': 1,
                                                                                                          'sigma': 1.6},
                                                                                           n_jobs=-1)),
                                                                                ('amplitude-11',
                                                                                 Amplitude(metric='heat',
                                                                                           metric_params={'n_bins': 100,
                                                                                                          'p': 1,
                                                                                                          'sigma': 3.2},
                                                                                           n_jobs=-1)),
                                                                                ('amplitude-12',
                                                                                 Amplitude(metric='heat',
                                                                                           metric_params={'n_bins': 100,
                                                                                                          'p': 2,
                                                                                                          'sigma': 1.6},
                                                                                           n_jobs=-1)),
                                                                                ('amplitude-13',
                                                                                 Amplitude(metric='heat',
                                                                                           metric_params={'n_bins': 100,
                                                                                                          'p': 2,
                                                                                                          'sigma': 3.2},
                                                                                           n_jobs=-1))]))]))])

In [15]:
diagrams = utils.alpha_persistence.AlphaComplexTransformer().fit_transform(X_train)

print(type(diagrams))
print(len(diagrams))

for i in range(3):
    print(diagrams[i].shape)

<class 'numpy.ndarray'>
400
(70102, 3)
(70102, 3)
(70102, 3)


In [17]:
for dim in [0, 1, 2]:
    counts = [(d[:, 2] == dim).sum() for d in diagrams]
    print(dim, min(counts), max(counts))

0 23527 56628
1 8795 30132
2 3676 18453


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    point_clouds,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

In [8]:
X_train_tda = tda_union.fit_transform(X_train)
X_train_tda.shape

ValueError: All persistence diagrams in the collection must have the same number of birth-death-dimension triples in any given homology dimension. This is not true in homology dimension 0.0. Trivial triples for which birth = death may be added or removed to fulfill this requirement.

In [230]:
from sklearn.ensemble import RandomForestClassifier


rf = RandomForestClassifier(n_jobs=-1)
rf.fit(X_train_tda, y_train)

X_test_tda = tda_union.transform(X_test)
rf.score(X_test_tda, y_test)

0.4

In [231]:
y_pred = rf.predict(X_test_tda)

In [232]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.33      0.33      0.33        40
           1       0.58      0.78      0.67        40
           2       0.35      0.33      0.34        40
           3       0.22      0.25      0.23        40
           4       0.39      0.38      0.38        40
           5       0.32      0.28      0.30        40
           6       0.71      0.60      0.65        40
           7       0.45      0.42      0.44        40
           8       0.27      0.30      0.29        40
           9       0.39      0.35      0.37        40

    accuracy                           0.40       400
   macro avg       0.40      0.40      0.40       400
weighted avg       0.40      0.40      0.40       400



In [237]:
from sklearn.pipeline import Pipeline

height_pipeline = Pipeline([
    ('binarizer', Binarizer(threshold=0.4)),
    ('filtration', HeightFiltration()),
    ('diagram', CubicalPersistence()),
    ('feature', PersistenceEntropy(nan_fill_value=-1)),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [ ]:
from sklearn.model_selection import GridSearchCV

direction_list = [[0, 1, 0], [0, 0, 1], [0, 1, 1], [1, 1, 1]]
n_estimators_list = [500, 1000, 2000]

param_grid = {
    "filtration__direction": [np.array(direction) for direction in direction_list],
    "classifier__n_estimators": [n_estimators for n_estimators in n_estimators_list],
}

grid_search = GridSearchCV(
    estimator=height_pipeline, param_grid=param_grid, cv=3, n_jobs=-1
)

grid_search.fit(X_train, y_train)